# 🧠 Final Lab Revision Master Assignment
## CS201L — AI Laboratory | IIT Dharwad | IS24BM003
### Complete Revision: Lab 2 → Lab 13 | Data Preprocessing → Clustering

---

> **HOW TO USE THIS NOTEBOOK:**
> - Run cells top-to-bottom on synthetic/built-in datasets — NO external files needed
> - Each section = 1 topic. Read the markdown, run the code, check the output
> - ⚠️ marks common exam mistakes | 💡 marks shortcuts/tips | 🎯 marks likely viva questions

---

## 📋 TABLE OF CONTENTS
1. [Setup & Imports](#setup)
2. [Data Preprocessing — EDA, IQR, Outliers](#lab2)
3. [Handling Missing Values](#lab3)
4. [Normalization & Standardization](#lab4a)
5. [Redundancy Removal — Correlation Analysis](#lab4b)
6. [Feature Selection & PCA](#lab5pca)
7. [Train/Val/Test Splitting (Stratified)](#split)
8. [KNN Classifier](#knn)
9. [Performance Metrics (Accuracy, Precision, Recall, F1, Confusion Matrix)](#metrics)
10. [Reference Template Helper Functions](#helpers)
11. [Bayes Classifier (from scratch)](#bayes)
12. [Naive Bayes Classifier (sklearn)](#naivebayes)
13. [Logistic Regression](#logreg)
14. [SVM — Linear, Polynomial, RBF Kernels](#svm)
15. [Perceptron Model (from scratch)](#perceptron)
16. [Neural Networks with PyTorch](#nn)
17. [Linear & Polynomial Regression + Ridge](#regression)
18. [Autoregression (AR Models + NN for Time Series)](#autoregression)
19. [Clustering — K-Means & K-Medoids](#kmeans)
20. [Clustering — Hierarchical (Agglomerative + Dendrogram)](#hierarchical)
21. [Clustering — DBSCAN](#dbscan)
22. [Clustering Metrics (Purity, NMI, Silhouette, Inertia)](#clustermetrics)
23. [1-Day Revision Strategy](#revision)


---
<a id='setup'></a>
## Section 1 — Setup & Imports

All code in this notebook uses **sklearn built-in datasets** (Iris, Wine, Breast Cancer, Digits, California Housing) so you can run everything without any CSV files.

> 💡 **Exam tip:** Always import everything at the top of your notebook. Copy this cell as your boilerplate.


In [ ]:
# ============================================================
# MASTER IMPORTS — copy this to every lab notebook
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

# Data / Splitting
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split

# Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, mean_squared_error,
    normalized_mutual_info_score, silhouette_score
)
from sklearn.preprocessing import PolynomialFeatures
import scipy.cluster.hierarchy as sch
from collections import Counter

# PyTorch (for Neural Networks)
import torch
import torch.nn as nn
import torch.optim as optim

# scipy
from scipy.stats import multivariate_normal

np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ All libraries imported successfully!")


---
<a id='lab2'></a>
## Section 2 — Exploratory Data Analysis, IQR & Outlier Detection
**(Labs 2 & 4 — residential_housing, energy_efficiency)**

### 📖 Theory
- **EDA** = understand your data before modelling: shape, dtypes, distributions, outliers
- **IQR (Interquartile Range)** = Q3 − Q1. Robust to outliers, unlike std.
- **Outlier rule**: point is an outlier if `x < Q1 − 1.5×IQR` or `x > Q3 + 1.5×IQR`
- **Five-number summary**: min, Q1, median, Q3, max

### 📐 Formulas
```
IQR = Q3 - Q1
Lower Fence = Q1 - 1.5 * IQR
Upper Fence = Q3 + 1.5 * IQR
```

### 🎯 Viva Questions
1. What is IQR and why is it preferred over std for outlier detection?
2. What does the boxplot whisker represent?
3. How do you detect outliers programmatically using IQR?
4. What is the difference between mean, median, and mode?

⚠️ **Common mistake:** Using `std` for outlier detection — prefer IQR since std is itself influenced by outliers.


In [ ]:
# ============================================================
# EDA, IQR, and Outlier Detection
# Using sklearn Wine dataset as a stand-in for any numeric dataset
# ============================================================

data = load_wine()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)


In [ ]:
# ---- Summary Statistics ----
print("=== Summary Statistics ===")
print(df.describe())


In [ ]:
# ---- Five Number Summary + IQR for one column ----
col = 'alcohol'

Q1 = df[col].quantile(0.25)
Q2 = df[col].quantile(0.50)   # median
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print(f"Five Number Summary for '{col}':")
print(f"  Min    : {df[col].min():.4f}")
print(f"  Q1     : {Q1:.4f}")
print(f"  Median : {Q2:.4f}")
print(f"  Q3     : {Q3:.4f}")
print(f"  Max    : {df[col].max():.4f}")
print(f"  IQR    : {IQR:.4f}")
print(f"  Lower Fence: {lower_fence:.4f}")
print(f"  Upper Fence: {upper_fence:.4f}")

outliers = df[(df[col] < lower_fence) | (df[col] > upper_fence)]
print(f"  Outliers ({col}): {len(outliers)} rows")
print(f"  Outlier % : {len(outliers)/len(df)*100:.2f}%")


In [ ]:
# ---- IQR for ALL numeric columns ----
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series[(series < lower) | (series > upper)]

numeric_cols = df.select_dtypes(include='number').columns.drop('target')
print("Outlier count per column:")
for col in numeric_cols:
    n = len(detect_outliers_iqr(df[col]))
    print(f"  {col:30s}: {n} outliers")


In [ ]:
# ---- Boxplot (before and after outlier correction) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Select 5 columns for visualization
cols_to_show = list(numeric_cols[:5])

# Before correction
df[cols_to_show].boxplot(ax=axes[0], rot=45)
axes[0].set_title('Boxplot — Before Outlier Correction')

# After correction: replace outliers with median
df_corrected = df.copy()
for col in numeric_cols:
    median = df[col].median()
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_corrected.loc[(df_corrected[col] < lower) | (df_corrected[col] > upper), col] = median

df_corrected[cols_to_show].boxplot(ax=axes[1], rot=45)
axes[1].set_title('Boxplot — After Outlier Correction (replace w/ median)')
plt.tight_layout()
plt.show()
print("✅ Outlier correction done — replaced outliers with column median")


In [ ]:
# ---- Histogram + KDE + ±2.7σ lines (Lab 2 style) ----
col = 'alcohol'
d = df[col].values
mu = d.mean()
sigma = d.std()
ucl = mu + 2.7 * sigma
lcl = mu - 2.7 * sigma

plt.figure(figsize=(10, 5))
plt.hist(d, bins=30, density=True, alpha=0.6, edgecolor='black', label='Histogram')
from scipy.stats import gaussian_kde
kde = gaussian_kde(d)
x_vals = np.linspace(d.min(), d.max(), 500)
plt.plot(x_vals, kde(x_vals), linewidth=2, label='KDE')
plt.axvline(mu,    color='black', linestyle='--', label=f'Mean={mu:.2f}')
plt.axvline(np.median(d), color='blue', linestyle='-.', label=f'Median={np.median(d):.2f}')
plt.axvline(ucl, color='red', linestyle=':', label=f'+2.7σ={ucl:.2f}')
plt.axvline(lcl, color='red', linestyle=':', label=f'-2.7σ={lcl:.2f}')
plt.title(f'Distribution of {col}')
plt.xlabel(col)
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()


---
<a id='lab3'></a>
## Section 3 — Handling Missing Values
**(Lab 3 — residential_housing)**

### 📖 Theory
Missing values occur when data collection fails. Strategies:
- **Drop rows** with too many missing values (threshold: 50% of columns)
- **Drop rows** where target variable is missing
- **Impute**: fill with mean, median, mode, or forward-fill (ffill)

### 📐 Decision Tree
```
Missing value?
├── > 50% of features missing? → Drop the row
├── Target column missing?     → Drop the row  
└── Otherwise → Impute (median preferred for skewed, mean for normal)
```

### 🎯 Viva Questions
1. Why do we prefer median over mean for imputation with skewed data?
2. What is forward fill (ffill)? When is it used?
3. What is `dropna(thresh=n)`?
4. Why should we NOT impute the target variable?

⚠️ **Common mistake:** Imputing BEFORE splitting into train/test. Always impute AFTER splitting (fit imputer on train only).


In [ ]:
# ============================================================
# Handling Missing Values
# We'll inject missing values into the Wine dataset to demonstrate
# ============================================================

df_missing = df.copy()

# Inject ~15% missing values randomly
np.random.seed(0)
mask = np.random.random(df_missing.shape) < 0.15
mask[:, -1] = False  # don't touch the target column
df_missing[mask] = np.nan

print("=== Missing Values Per Column ===")
missing_counts = df_missing.isnull().sum()
print(missing_counts)
print(f"\nTotal missing values: {missing_counts.sum()}")


In [ ]:
# ---- Missing values per row distribution ----
missing_per_row = df_missing.isnull().sum(axis=1)
print("Missing values per row (first 20 rows):")
print(missing_per_row.value_counts().sort_index())

# Bar plot
plt.figure(figsize=(8, 4))
missing_per_row.value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.xlabel('# Missing Values in Row')
plt.ylabel('# Rows')
plt.title('Distribution of Missing Values Per Row')
plt.tight_layout()
plt.show()


In [ ]:
# ---- Step 1: Drop rows with > 50% missing features ----
n_cols = df_missing.shape[1] - 1  # exclude target
threshold_50pct = int(n_cols * 0.5)
df_cleaned = df_missing.dropna(thresh=n_cols - threshold_50pct + 1)
print(f"Rows before: {len(df_missing)} | After dropping >50% missing: {len(df_cleaned)}")

# ---- Step 2: Drop rows where TARGET is missing ----
df_cleaned = df_cleaned.dropna(subset=['target'])
print(f"After dropping rows with missing target: {len(df_cleaned)}")

# ---- Step 3a: Impute with MEDIAN (recommended for skewed data) ----
df_imputed_median = df_cleaned.copy()
numeric_cols_clean = df_cleaned.select_dtypes(include='number').columns
medians = df_cleaned[numeric_cols_clean].median()
df_imputed_median[numeric_cols_clean] = df_imputed_median[numeric_cols_clean].fillna(medians)
print(f"\nMissing after median imputation: {df_imputed_median.isnull().sum().sum()}")

# ---- Step 3b: Forward Fill (ffill) ----
df_ffill = df_cleaned.fillna(method='ffill')
print(f"Missing after ffill: {df_ffill.isnull().sum().sum()}")
print("\n✅ Missing value handling complete!")


In [ ]:
# ---- Compare stats before/after imputation ----
print("=== Stats comparison (median imputation) ===")
comparison = pd.DataFrame({
    'Before (median)': df_cleaned[numeric_cols_clean].median(),
    'After  (median)': df_imputed_median[numeric_cols_clean].median()
})
print(comparison.to_string())


---
<a id='lab4a'></a>
## Section 4 — Normalization & Standardization
**(Lab 4 — energy_efficiency)**

### 📖 Theory
Raw features often have very different scales. This hurts distance-based and gradient-based algorithms.

| Method | Formula | Range | When to use |
|--------|---------|-------|-------------|
| **Min-Max Normalization** | `(x - min) / (max - min)` | [0, 1] | Neural nets, KNN |
| **Min-Max (custom range)** | `a + (x-min)*(b-a)/(max-min)` | [a, b] | When specific range needed |
| **Z-score Standardization** | `(x - μ) / σ` | ~[-3, 3] | SVM, Logistic Regression, PCA |

### 📐 Key Formulas
```
Min-Max [0,1]: x_scaled = (x - x_min) / (x_max - x_min)
Z-score:       x_scaled = (x - μ) / σ
```

### 🎯 Viva Questions
1. What is the difference between normalization and standardization?
2. Why must you fit the scaler on TRAIN and only transform on val/test?
3. When would you prefer standardization over normalization?
4. What happens to outliers after min-max normalization?

⚠️ **Common mistake:** Fitting the scaler on the FULL dataset (data leakage). Always `fit` only on training data!


In [ ]:
# ============================================================
# Normalization & Standardization
# ============================================================

iris = load_iris()
X_raw = pd.DataFrame(iris.data, columns=iris.feature_names)
print("Raw data stats:")
print(X_raw.describe().loc[['min', 'max', 'mean', 'std']])


In [ ]:
# ---- Min-Max Normalization [0, 1] ----
scaler_01 = MinMaxScaler(feature_range=(0, 1))
X_norm_01 = scaler_01.fit_transform(X_raw)
df_norm_01 = pd.DataFrame(X_norm_01, columns=X_raw.columns)

print("=== After Min-Max Normalization [0,1] ===")
print(df_norm_01.describe().loc[['min', 'max', 'mean', 'std']])


In [ ]:
# ---- Min-Max Normalization [10, 30] (custom range like Lab 4) ----
scaler_custom = MinMaxScaler(feature_range=(10, 30))
X_norm_custom = scaler_custom.fit_transform(X_raw)
df_norm_custom = pd.DataFrame(X_norm_custom, columns=X_raw.columns)

print("=== After Min-Max Normalization [10, 30] ===")
print(df_norm_custom.describe().loc[['min', 'max', 'mean', 'std']])


In [ ]:
# ---- Z-score Standardization ----
scaler_std = StandardScaler()
X_std = scaler_std.fit_transform(X_raw)
df_std = pd.DataFrame(X_std, columns=X_raw.columns)

print("=== After Z-score Standardization ===")
print(df_std.describe().loc[['min', 'max', 'mean', 'std']])


In [ ]:
# ---- Visual comparison ----
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
col = 'sepal length (cm)'

axes[0].hist(X_raw[col], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Original')
axes[0].set_xlabel(col)

axes[1].hist(df_norm_01[col], bins=20, color='orange', edgecolor='black')
axes[1].set_title('Min-Max [0,1]')
axes[1].set_xlabel(col)

axes[2].hist(df_std[col], bins=20, color='green', edgecolor='black')
axes[2].set_title('Z-score (Standardized)')
axes[2].set_xlabel(col)

plt.suptitle('Effect of Scaling on Distribution', fontsize=13)
plt.tight_layout()
plt.show()

# KEY INSIGHT: shape of distribution does NOT change, only the scale
print("\n💡 Note: Scaling changes the RANGE, NOT the shape of the distribution!")


---
<a id='lab4b'></a>
## Section 5 — Redundancy Removal via Correlation Analysis
**(Lab 4 — energy_efficiency)**

### 📖 Theory
Highly correlated features carry redundant information → remove one of the pair.

| Method | Range | Formula |
|--------|-------|---------|
| **Pearson** | [-1, 1] | `r = Σ(xi-x̄)(yi-ȳ) / (n·σx·σy)` — linear relationships |
| **Spearman** | [-1, 1] | Rank-based — monotonic (non-linear) relationships |

**Rule of thumb:** If |r| > 0.9 between two features, consider dropping one.

### 🎯 Viva Questions
1. What is the difference between Pearson and Spearman correlation?
2. How do you identify redundant features?
3. What does a correlation heatmap tell you?
4. Why is removing redundant features important?

⚠️ **Common mistake:** Removing a feature just because it has low correlation with another feature. Only remove if it's also low correlation with the target.


In [ ]:
# ============================================================
# Correlation Analysis — Redundancy Removal
# ============================================================

df_wine = pd.DataFrame(load_wine().data, columns=load_wine().feature_names)
df_wine['target'] = load_wine().target

# --- Pearson Correlation Matrix ---
pearson_corr = df_wine.corr(method='pearson')
spearman_corr = df_wine.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt='.2f', ax=axes[0],
            annot_kws={'size': 7})
axes[0].set_title('Pearson Correlation Heatmap', fontsize=13)

sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt='.2f', ax=axes[1],
            annot_kws={'size': 7})
axes[1].set_title('Spearman Correlation Heatmap', fontsize=13)

plt.tight_layout()
plt.show()


In [ ]:
# --- Feature correlation with target ---
target_corr_pearson  = df_wine.corr(method='pearson')['target'].drop('target')
target_corr_spearman = df_wine.corr(method='spearman')['target'].drop('target')

comparison = pd.DataFrame({
    'Pearson with Target': target_corr_pearson,
    'Spearman with Target': target_corr_spearman
}).sort_values('Pearson with Target')

print("Feature correlations with target:")
print(comparison.to_string())

# Bar plot
comparison.plot(kind='bar', figsize=(12, 5))
plt.title('Feature Correlation with Target (Pearson vs Spearman)')
plt.ylabel('Correlation Coefficient')
plt.tight_layout()
plt.show()


In [ ]:
# --- Identify highly correlated feature PAIRS (redundant) ---
threshold = 0.85
corr_matrix = df_wine[df_wine.columns.drop('target')].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

redundant_pairs = [(col, row) for col in upper.columns 
                   for row in upper.index if upper[col][row] > threshold]

print(f"Highly correlated pairs (|r| > {threshold}):")
for pair in redundant_pairs:
    print(f"  {pair[0]:30s} <-> {pair[1]:30s}  r={corr_matrix[pair[0]][pair[1]]:.4f}")


---
<a id='lab5pca'></a>
## Section 6 — Feature Selection & PCA (Principal Component Analysis)
**(Lab 5 — Activity Detection)**

### 📖 Theory
PCA finds directions (principal components) of maximum variance in data.
- Reduces dimensionality while retaining maximum information
- PCs are orthogonal (uncorrelated)
- Eigenvalues → variance explained by each PC

### 📐 Key Steps
1. Standardize features (Z-score)
2. Compute covariance matrix
3. Find eigenvalues and eigenvectors
4. Sort by eigenvalue (descending)
5. Select k components explaining ≥ desired variance
6. Transform data: `Z = X × W` where W = top-k eigenvectors

### 📐 Formulas
```
Explained variance ratio of PC_i = λ_i / Σλ_j
Cumulative variance at k PCs = Σ(i=1 to k) λ_i / Σλ_j
```

### 🎯 Viva Questions
1. What does "explained variance ratio" mean in PCA?
2. Why must we standardize before PCA?
3. What is the difference between PCA with all components vs 99% variance?
4. What does a scree plot show?
5. Are the principal components interpretable?

⚠️ **Common mistake:** Applying PCA BEFORE splitting into train/test, OR fitting PCA on test data.


In [ ]:
# ============================================================
# PCA — Principal Component Analysis
# Using Digits dataset (64 features → good for dimensionality reduction)
# ============================================================

digits = load_digits()
X_digits = digits.data  # 64 features
y_digits = digits.target

# Step 1: Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_digits)

print(f"Original shape: {X_scaled.shape}")


In [ ]:
# ---- PCA with ALL components ----
pca_all = PCA()
pca_all.fit(X_scaled)

# Scree plot
explained = pca_all.explained_variance_ratio_
cumulative = np.cumsum(explained)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(explained)+1), explained, 'bo-', markersize=4)
plt.xlabel('Component Number')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot — Individual Variance')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative)+1), cumulative, 'go-', markersize=4)
plt.axhline(0.99, color='red', linestyle='--', label='99% variance')
plt.axhline(0.95, color='orange', linestyle='--', label='95% variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance Explained')
plt.title('Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

n_99 = np.argmax(cumulative >= 0.99) + 1
n_95 = np.argmax(cumulative >= 0.95) + 1
print(f"Components for 99% variance: {n_99}")
print(f"Components for 95% variance: {n_95}")
print(f"Original features: {X_scaled.shape[1]}")
print(f"Reduction (99%): {X_scaled.shape[1]} → {n_99} features")


In [ ]:
# ---- PCA with 99% variance ----
pca_99 = PCA(n_components=0.99)
X_pca_99 = pca_99.fit_transform(X_scaled)

print(f"=== PCA (99% variance) ===")
print(f"Input shape : {X_scaled.shape}")
print(f"Output shape: {X_pca_99.shape}")
print(f"n_components: {pca_99.n_components_}")
print(f"Total variance explained: {pca_99.explained_variance_ratio_.sum():.4f}")


In [ ]:
# ---- 2D PCA Visualization ----
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_digits, cmap='tab10', alpha=0.7, s=15)
plt.colorbar(scatter, label='Digit Class')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA Projection — Digits Dataset')
plt.tight_layout()
plt.show()

print(f"Variance explained by PC1: {pca_2d.explained_variance_ratio_[0]:.4f}")
print(f"Variance explained by PC2: {pca_2d.explained_variance_ratio_[1]:.4f}")
print(f"Total (2D): {pca_2d.explained_variance_ratio_.sum():.4f}")


---
<a id='split'></a>
## Section 7 — Train / Validation / Test Split (Stratified)
**(Lab 5 — Activity Detection)**

### 📖 Theory
- **Training set (~60%)**: fit the model
- **Validation set (~20%)**: tune hyperparameters
- **Test set (~20%)**: final unbiased evaluation

**Stratified split** = maintains class proportions in each subset

### 📐 Steps
```python
# Step 1: 60/40 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y)
# Step 2: Split temp 50/50 → 20% val, 20% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)
```

### 🎯 Viva Questions
1. Why do we use stratified splitting?
2. What is the purpose of a validation set (vs test set)?
3. What is data leakage?

⚠️ **Common mistake:** Fitting any preprocessor (scaler, PCA) on the FULL dataset before splitting. Always split FIRST, then fit on train only.


In [ ]:
# ============================================================
# Stratified Train / Val / Test Split
# ============================================================

X = iris.data
y = iris.target

# Step 1: 60% train, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42)

# Step 2: 50% of temp → val (20%), 50% → test (20%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"Train size      : {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation size : {X_val.shape[0]} ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test size       : {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")

print("\nClass distribution (should be proportional):")
print(f"  Train  : {np.bincount(y_train)}")
print(f"  Val    : {np.bincount(y_val)}")
print(f"  Test   : {np.bincount(y_test)}")

print("\n✅ Stratification verified — classes proportional across all splits!")


---
<a id='knn'></a>
## Section 8 — KNN (K-Nearest Neighbors)
**(Lab 5 — Activity Detection)**

### 📖 Theory
KNN classifies a new point by majority vote of its K nearest neighbors.
- **Lazy learner** — no training, just memorizes data
- Distance metric: usually Euclidean

### 📐 Formula
```
Euclidean distance: d(x,y) = √Σ(xi - yi)²
Prediction: class = mode of labels of K nearest neighbors
```

### 🎯 Viva Questions
1. What is the effect of large K vs small K?
2. Why must we normalize before KNN?
3. What is the curse of dimensionality?
4. Is KNN a parametric or non-parametric model?
5. How do we choose the best K?

⚠️ **Common mistake:** Using KNN without scaling. Large-scale features dominate distance calculations.

💡 **Tip:** Plot validation accuracy vs K (odd values 1-20) to find the best K.


In [ ]:
# ============================================================
# KNN Classifier with hyperparameter tuning
# ============================================================

# Standardize (required for KNN)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Try different values of K
k_values = list(range(1, 21))
val_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    acc = accuracy_score(y_val, knn.predict(X_val_s))
    val_accuracies.append(acc)

best_k = k_values[np.argmax(val_accuracies)]
print(f"Best K on validation set: K={best_k} (accuracy={max(val_accuracies):.4f})")

# Plot
plt.figure(figsize=(10, 4))
plt.plot(k_values, val_accuracies, 'bo-', markersize=6)
plt.axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K')
plt.ylabel('Validation Accuracy')
plt.title('KNN: Validation Accuracy vs K')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Train with best K and evaluate ----
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_s, y_train)

y_val_pred  = knn_best.predict(X_val_s)
y_test_pred = knn_best.predict(X_test_s)

print(f"KNN (K={best_k}) Results:")
print(f"  Val  Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"  Test Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")


---
<a id='metrics'></a>
## Section 9 — Performance Metrics
**(Used in ALL Labs 5–13)**

### 📖 Theory — Confusion Matrix
```
              Predicted
              Pos   Neg
Actual  Pos [ TP  | FN ]
        Neg [ FP  | TN ]
```

| Metric | Formula | Description |
|--------|---------|-------------|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Overall correct predictions |
| **Precision** | TP/(TP+FP) | Of predicted positives, how many are actually positive |
| **Recall** | TP/(TP+FN) | Of actual positives, how many did we catch |
| **F1-Score** | 2×P×R/(P+R) | Harmonic mean of precision and recall |

### Macro vs Micro Averaging (multiclass)
- **Macro**: compute metric for each class, take simple average (treats classes equally)
- **Micro**: aggregate TP/FP/FN across all classes, then compute (favors larger classes)

### 🎯 Viva Questions
1. When would you prefer Recall over Precision? (e.g., cancer detection)
2. What does a diagonal confusion matrix mean?
3. Difference between macro and micro F1?
4. When is accuracy misleading? (class imbalance)
5. What is micro-precision equal to in multiclass? (= accuracy)


In [ ]:
# ============================================================
# Complete Performance Metrics Template (from Lab 6, 7, 8)
# This is the REFERENCE FUNCTION used across all labs
# ============================================================

def print_metrics(y_true, y_pred, dataset_name="Test"):
    """
    Reference template for computing and printing all metrics.
    Used in Labs 6, 7, 8 — copy this into any exam notebook.
    """
    cm       = confusion_matrix(y_true, y_pred)
    acc      = accuracy_score(y_true, y_pred)
    prec_mac = precision_score(y_true, y_pred, average='macro',  zero_division=0)
    prec_mic = precision_score(y_true, y_pred, average='micro',  zero_division=0)
    rec_mac  = recall_score(y_true, y_pred,    average='macro',  zero_division=0)
    rec_mic  = recall_score(y_true, y_pred,    average='micro',  zero_division=0)
    f1_mac   = f1_score(y_true, y_pred,        average='macro',  zero_division=0)
    f1_mic   = f1_score(y_true, y_pred,        average='micro',  zero_division=0)

    print(f"\n{'='*50}")
    print(f"  {dataset_name} Results")
    print(f"{'='*50}")
    print("Confusion Matrix:")
    print(cm)
    print(f"\nAccuracy          : {acc:.4f}")
    print(f"Precision (Macro) : {prec_mac:.4f}")
    print(f"Precision (Micro) : {prec_mic:.4f}")
    print(f"Recall    (Macro) : {rec_mac:.4f}")
    print(f"Recall    (Micro) : {rec_mic:.4f}")
    print(f"F1-Score  (Macro) : {f1_mac:.4f}")
    print(f"F1-Score  (Micro) : {f1_mic:.4f}")
    return acc


def plot_confusion_matrix(y_true, y_pred, title, classes=None):
    """Plot a nicely formatted confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)
    plt.title(title, fontsize=13)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    return cm


print("✅ Reference metric functions defined!")
print_metrics(y_test, y_test_pred, "KNN Test Set")


In [ ]:
# ---- Confusion matrix visualization ----
classes = [str(c) for c in iris.target_names]
plot_confusion_matrix(y_test, y_test_pred, 
                      f'KNN (K={best_k}) — Confusion Matrix', 
                      classes=classes)


---
<a id='bayes'></a>
## Section 10 — Bayes Classifier (From Scratch) & Naive Bayes (sklearn)
**(Lab 6 — HAR / Date Fruit)**

### 📖 Theory

**Bayes' Theorem:**
```
P(class | X) ∝ P(X | class) × P(class)
```
- **Prior** P(class): frequency of class in training data
- **Likelihood** P(X|class): probability of observing X given the class
- **Posterior** P(class|X): what we want

**Bayes Classifier (Gaussian):**
- Models P(X|class) as a multivariate Gaussian
- Full covariance matrix per class
- Can handle correlated features

**Naive Bayes:**
- ASSUMES features are conditionally independent
- Uses univariate Gaussian per feature per class
- Much faster, works well despite the independence assumption

### 📐 Formula
```
Gaussian likelihood: P(X|class) = (1/√(2π)^d|Σ|) × exp(-0.5(X-μ)ᵀΣ⁻¹(X-μ))
```

### 🎯 Viva Questions
1. What is the Naive Bayes independence assumption?
2. When can Naive Bayes outperform more complex models?
3. What is the prior probability?
4. Why is Bayes classifier called "optimal" (MAP classifier)?

⚠️ **Common mistake:** Forgetting to handle singular covariance matrices (use `allow_singular=True`).


In [ ]:
# ============================================================
# Bayes Classifier — From Scratch (Multivariate Gaussian)
# Matches exactly the Lab 6 BayesClassifier class
# ============================================================

class BayesClassifier:
    """
    Gaussian Bayes Classifier.
    Models each class as a multivariate Gaussian.
    """
    
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.cls_mean  = {}
        self.cls_cov   = {}
        self.cls_prior = {}
        
        for c in self.classes:
            X_c = X[y == c]
            self.cls_mean[c]  = np.mean(X_c, axis=0)
            self.cls_cov[c]   = np.cov(X_c, rowvar=False)
            self.cls_prior[c] = X_c.shape[0] / X.shape[0]
        
        return self
    
    def predict(self, X):
        posteriors = []
        for c in self.classes:
            likelihood = multivariate_normal.pdf(
                X,
                mean=self.cls_mean[c],
                cov=self.cls_cov[c],
                allow_singular=True   # handles singular covariance
            )
            posterior = likelihood * self.cls_prior[c]
            posteriors.append(posterior)
        
        posteriors = np.array(posteriors)  # shape: (n_classes, n_samples)
        return self.classes[np.argmax(posteriors, axis=0)]


# ---- Use with PCA-reduced Iris data ----
X_iris = iris.data
y_iris = iris.target

X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, 
                                            stratify=y_iris, random_state=42)

# Apply PCA (like Lab 6 used scaled + PCA variants)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

pca = PCA(n_components=0.99)
X_tr_pca = pca.fit_transform(X_tr_s)
X_te_pca = pca.transform(X_te_s)

# Fit Bayes
bayes = BayesClassifier()
bayes.fit(X_tr_pca, y_tr)
y_pred_bayes = bayes.predict(X_te_pca)

print("=== Bayes Classifier (from scratch) ===")
print_metrics(y_te, y_pred_bayes, "Bayes — Test")


In [ ]:
# ============================================================
# Naive Bayes — sklearn GaussianNB
# ============================================================

gnb = GaussianNB()
gnb.fit(X_tr_pca, y_tr)
y_pred_gnb = gnb.predict(X_te_pca)

print("=== Naive Bayes (sklearn GaussianNB) ===")
print_metrics(y_te, y_pred_gnb, "Naive Bayes — Test")

# Compare
acc_bayes = accuracy_score(y_te, y_pred_bayes)
acc_gnb   = accuracy_score(y_te, y_pred_gnb)
print(f"\nComparison:")
print(f"  Bayes (scratch) accuracy: {acc_bayes:.4f}")
print(f"  Naive Bayes accuracy    : {acc_gnb:.4f}")
print("  Note: Naive Bayes often matches or beats full Bayes due to regularization effect")


---
<a id='logreg'></a>
## Section 11 — Logistic Regression
**(Lab 7 — Activity Detection, original + scaled + PCA variants)**

### 📖 Theory
Logistic Regression models the **probability** of a class using the sigmoid function.
Despite the name, it's a **classification** algorithm.

### 📐 Formulas
```
Sigmoid: σ(z) = 1 / (1 + e^(-z))
Linear combination: z = w₀ + w₁x₁ + ... + wₙxₙ
Output: P(y=1|x) = σ(z)
Decision: y = 1 if P(y=1|x) > 0.5, else y = 0
Cost: J = -1/N × Σ [y log(ŷ) + (1-y) log(1-ŷ)]  (cross-entropy)
```

### 🎯 Viva Questions
1. Why is it called "regression" if it does classification?
2. What is the sigmoid function? Why is it used?
3. What is the role of the regularization parameter C in sklearn?
4. Why use `solver='liblinear'` for smaller datasets?
5. What is the difference between binary and multiclass logistic regression?

⚠️ **Common mistake:** Not standardizing before Logistic Regression with regularization.


In [ ]:
# ============================================================
# Logistic Regression — Original, Scaled, PCA variants (Lab 7 pattern)
# ============================================================

X_iris = iris.data
y_iris = iris.target

X_train, X_temp, y_train, y_temp = train_test_split(
    X_iris, y_iris, test_size=0.4, stratify=y_iris, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# ---- Variant 1: Original (unscaled) data ----
lr_orig = LogisticRegression(solver='liblinear', max_iter=1000)
lr_orig.fit(X_train, y_train)

print("=== TASK 1.1: Logistic Regression — ORIGINAL DATA ===")
print_metrics(y_val, lr_orig.predict(X_val), "Validation")
print_metrics(y_test, lr_orig.predict(X_test), "Test")


In [ ]:
# ---- Variant 2: Scaled data ----
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_vl_s = scaler.transform(X_val)
X_te_s = scaler.transform(X_test)

lr_scaled = LogisticRegression(solver='liblinear', max_iter=1000)
lr_scaled.fit(X_tr_s, y_train)

print("=== TASK 2.1: Logistic Regression — SCALED DATA ===")
print_metrics(y_val, lr_scaled.predict(X_vl_s), "Validation")
print_metrics(y_test, lr_scaled.predict(X_te_s), "Test")


In [ ]:
# ---- Variant 3: PCA-reduced data (99% variance) ----
pca = PCA(n_components=0.99)
X_tr_pca = pca.fit_transform(X_tr_s)
X_vl_pca = pca.transform(X_vl_s)
X_te_pca = pca.transform(X_te_s)

print(f"PCA: {X_tr_s.shape[1]} features → {X_tr_pca.shape[1]} PCs (99% variance)")

lr_pca = LogisticRegression(solver='liblinear', max_iter=1000)
lr_pca.fit(X_tr_pca, y_train)

print("=== TASK 4.1: Logistic Regression — PCA 99% Variance ===")
print_metrics(y_val, lr_pca.predict(X_vl_pca), "Validation")
print_metrics(y_test, lr_pca.predict(X_te_pca), "Test")


---
<a id='svm'></a>
## Section 12 — SVM (Support Vector Machine)
**(Lab 7 — Activity Detection)**

### 📖 Theory
SVM finds the **maximum-margin hyperplane** separating classes.
- **Support vectors**: data points closest to the decision boundary
- **C parameter**: regularization — large C = narrow margin, fits training data tightly
- **Kernel trick**: maps data to higher-dimensional space implicitly

### 📐 Kernel Functions
| Kernel | Formula | When to use |
|--------|---------|-------------|
| **Linear** | `K(x,y) = xᵀy` | Linearly separable data |
| **Polynomial** | `K(x,y) = (γxᵀy + r)^d` | Non-linear, controlled complexity |
| **RBF/Gaussian** | `K(x,y) = exp(-γ‖x-y‖²)` | General purpose, most common |

### 🎯 Viva Questions
1. What are support vectors?
2. What is the kernel trick?
3. What does the C parameter control?
4. Why is SVM sensitive to feature scaling?
5. How does SVM handle multiclass classification? (OvO or OvR)

⚠️ **Common mistake:** Using polynomial SVM without scaling — blows up numerically.


In [ ]:
# ============================================================
# SVM — Linear, Polynomial (degree sweep), RBF
# ============================================================

# Use digits dataset (more interesting than Iris for SVM)
X_tr_s = scaler.fit_transform(X_train)
X_vl_s = scaler.transform(X_val)
X_te_s = scaler.transform(X_test)

# ---- Task 1.2: Linear SVM ----
linear_svm = SVC(kernel='linear', C=1.0)
linear_svm.fit(X_tr_s, y_train)

print("=== SVM (LINEAR KERNEL) ===")
print_metrics(y_val,  linear_svm.predict(X_vl_s), "Validation")
print_metrics(y_test, linear_svm.predict(X_te_s), "Test")


In [ ]:
# ---- Task 1.3: Polynomial SVM — sweep over degrees ----
best_degree   = None
best_val_acc  = 0
degree_accs   = {}

for degree in [2, 3, 4, 5]:
    poly_svm = SVC(kernel='poly', degree=degree, C=1.0, gamma='scale')
    poly_svm.fit(X_tr_s, y_train)
    val_acc = poly_svm.score(X_vl_s, y_val)
    degree_accs[degree] = val_acc
    print(f"Degree={degree}, Val Accuracy={val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_degree  = degree

print(f"\nBest Degree: {best_degree} (Val Acc: {best_val_acc:.4f})")

# Plot
plt.figure(figsize=(7, 4))
plt.plot(list(degree_accs.keys()), list(degree_accs.values()), 'bo-', markersize=8)
plt.title('Polynomial SVM: Validation Accuracy vs Degree')
plt.xlabel('Degree')
plt.ylabel('Validation Accuracy')
plt.xticks([2, 3, 4, 5])
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Final Polynomial SVM with best degree ----
poly_svm_best = SVC(kernel='poly', degree=best_degree, C=1.0, gamma='scale')
poly_svm_best.fit(X_tr_s, y_train)

print(f"=== SVM (POLY degree={best_degree}) — Final Evaluation ===")
print_metrics(y_test, poly_svm_best.predict(X_te_s), "Test")

# ---- RBF SVM ----
rbf_svm = SVC(kernel='rbf', C=1.0, gamma='scale')
rbf_svm.fit(X_tr_s, y_train)
print("\n=== SVM (RBF KERNEL) ===")
print_metrics(y_test, rbf_svm.predict(X_te_s), "Test")


---
<a id='perceptron'></a>
## Section 13 — Perceptron Model (From Scratch)
**(Lab 8 background / Neural Network building block)**

### 📖 Theory
The Perceptron is the simplest neural network — a single artificial neuron.

### 📐 Algorithm
```
1. Initialize weights w = 0 (or random small values)
2. For each training sample (x, y):
   a. Compute: ŷ = step(w·x + b)  where step(z) = 1 if z≥0 else 0
   b. Update:  w += α × (y - ŷ) × x
               b += α × (y - ŷ)
3. Repeat until convergence or max epochs
```

### 🎯 Viva Questions
1. What activation function does the perceptron use?
2. Is a perceptron a linear classifier?
3. What is the perceptron convergence theorem?
4. What problem can a perceptron NOT solve? (XOR)


In [ ]:
# ============================================================
# Perceptron — From Scratch (Binary Classification)
# ============================================================

class Perceptron:
    """Simple Perceptron for binary classification."""
    
    def __init__(self, lr=0.01, n_epochs=1000):
        self.lr = lr
        self.n_epochs = n_epochs
    
    def fit(self, X, y):
        # Initialize weights
        n_features = X.shape[1]
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.errors_per_epoch = []
        
        for epoch in range(self.n_epochs):
            errors = 0
            for xi, yi in zip(X, y):
                y_pred = self._predict_single(xi)
                update = self.lr * (yi - y_pred)
                self.w += update * xi
                self.b += update
                errors += int(update != 0)
            self.errors_per_epoch.append(errors)
            if errors == 0:
                print(f"Converged at epoch {epoch+1}")
                break
        return self
    
    def _predict_single(self, x):
        return 1 if (np.dot(x, self.w) + self.b) >= 0 else 0
    
    def predict(self, X):
        return np.array([self._predict_single(xi) for xi in X])


# ---- Binary problem: Iris classes 0 vs 1 only ----
mask = y_iris <= 1
X_bin = iris.data[mask]
y_bin = iris.target[mask]

scaler_bin = StandardScaler()
X_bin_s = scaler_bin.fit_transform(X_bin)

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_bin_s, y_bin, test_size=0.2, random_state=42)

perceptron = Perceptron(lr=0.1, n_epochs=1000)
perceptron.fit(X_tr_b, y_tr_b)

y_pred_p = perceptron.predict(X_te_b)
print(f"\nPerceptron Test Accuracy: {accuracy_score(y_te_b, y_pred_p):.4f}")

# Plot convergence
plt.figure(figsize=(8, 4))
plt.plot(perceptron.errors_per_epoch[:50])
plt.xlabel('Epoch')
plt.ylabel('Misclassifications')
plt.title('Perceptron Convergence (Errors per Epoch)')
plt.grid(True)
plt.tight_layout()
plt.show()


---
<a id='nn'></a>
## Section 14 — Neural Networks with PyTorch (SGD)
**(Lab 8 — HAR dataset, multiclass)**

### 📖 Theory
A feedforward neural network with:
- Input layer → Hidden layer(s) with ReLU → Output layer (logits)
- Loss: CrossEntropyLoss
- Optimizer: SGD (used in labs)

### 📐 Architecture Pattern (from Lab 8)
```
Input (n_features) → Linear → ReLU → Linear → ReLU → ... → Linear → Output (n_classes)
```

### Convergence Criterion (Lab 8 pattern):
Stop early if `|loss(t) - loss(t-1)| < threshold` for `patience` consecutive epochs.

### 🎯 Viva Questions
1. What is the role of activation functions (ReLU)?
2. What is cross-entropy loss?
3. What is backpropagation?
4. What is the difference between SGD and Adam?
5. What is overfitting and how is it detected?

⚠️ **Common mistake:** Not calling `model.train()` before training and `model.eval()` before inference.


In [ ]:
# ============================================================
# Neural Network — PyTorch (from Lab 8 pattern)
# ============================================================

# Use Wine dataset
wine = load_wine()
X_wine = wine.data.astype(np.float32)
y_wine = wine.target

X_tr, X_te, y_tr, y_te = train_test_split(
    X_wine, y_wine, test_size=0.2, stratify=y_wine, random_state=42)

scaler_nn = StandardScaler()
X_tr_s = scaler_nn.fit_transform(X_tr).astype(np.float32)
X_te_s = scaler_nn.transform(X_te).astype(np.float32)

# Convert to tensors
X_train_t = torch.tensor(X_tr_s)
y_train_t = torch.tensor(y_tr, dtype=torch.long)
X_test_t  = torch.tensor(X_te_s)
y_test_t  = torch.tensor(y_te, dtype=torch.long)

print(f"Input features : {X_tr_s.shape[1]}")
print(f"Classes        : {len(np.unique(y_tr))}")
print(f"Train samples  : {len(X_train_t)}")


# ============================================================
# Model Definition — Configurable hidden layers
# ============================================================
class NNModel(nn.Module):
    """Feedforward NN with configurable hidden layers (from Lab 8)."""
    
    def __init__(self, input_size, hidden_layers, output_size):
        super().__init__()
        layers = []
        in_size = input_size
        for h in hidden_layers:
            layers.append(nn.Linear(in_size, h))
            layers.append(nn.ReLU())
            in_size = h
        layers.append(nn.Linear(in_size, output_size))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


def train_model(model, X, y, lr=0.01, max_epochs=1000, patience=10, threshold=1e-3):
    """Train with SGD + CrossEntropy. Early stopping on convergence."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    loss_history = []
    prev_loss = float('inf')
    counter = 0
    
    for epoch in range(1, max_epochs + 1):
        model.train()           # ← IMPORTANT
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        train_loss = loss.item()
        loss_history.append(train_loss)
        
        # Convergence check
        if abs(train_loss - prev_loss) < threshold:
            counter += 1
        else:
            counter = 0
        prev_loss = train_loss
        
        if counter >= patience:
            print(f"  Converged at epoch {epoch} | Loss: {train_loss:.4f}")
            break
        if epoch % 200 == 0:
            print(f"  Epoch {epoch} | Loss: {train_loss:.4f}")
    
    return loss_history


def evaluate_nn(model, X, y):
    """Evaluate model, return predictions."""
    model.eval()                # ← IMPORTANT
    with torch.no_grad():
        outputs = model(X)
        preds = torch.argmax(outputs, dim=1)
    return preds.numpy()


print("\n✅ NN classes and helper functions defined!")


In [ ]:
# ---- Train and compare different architectures ----
n_input   = X_tr_s.shape[1]
n_classes = len(np.unique(y_tr))

architectures = [
    {'name': 'Small  (64)',        'hidden': [64]},
    {'name': 'Medium (128, 64)',   'hidden': [128, 64]},
    {'name': 'Large  (256, 128)',  'hidden': [256, 128]},
]

results = {}
torch.manual_seed(42)

for arch in architectures:
    print(f"\nTraining: {arch['name']}")
    model = NNModel(n_input, arch['hidden'], n_classes)
    losses = train_model(model, X_train_t, y_train_t, lr=0.01, max_epochs=1000)
    
    preds_tr = evaluate_nn(model, X_train_t, y_train_t)
    preds_te = evaluate_nn(model, X_test_t,  y_test_t)
    
    acc_tr = accuracy_score(y_tr, preds_tr)
    acc_te = accuracy_score(y_te, preds_te)
    results[arch['name']] = {'train': acc_tr, 'test': acc_te, 'losses': losses}
    print(f"  Train Acc: {acc_tr:.4f} | Test Acc: {acc_te:.4f}")


In [ ]:
# ---- Loss convergence plots ----
plt.figure(figsize=(10, 4))
for name, res in results.items():
    plt.plot(res['losses'][:200], label=name)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.title('Training Loss Convergence (first 200 epochs)')
plt.legend()
plt.tight_layout()
plt.show()

# ---- Summary table ----
print("\n=== Architecture Comparison ===")
print(f"{'Architecture':25s} {'Train Acc':>12} {'Test Acc':>12}")
print('-' * 52)
for name, res in results.items():
    print(f"{name:25s} {res['train']:>12.4f} {res['test']:>12.4f}")


In [ ]:
# ---- Full metrics for best model ----
best_model_name = max(results, key=lambda k: results[k]['test'])
print(f"\nBest Architecture: {best_model_name}")

torch.manual_seed(42)
best_arch = next(a for a in architectures if a['name'] == best_model_name)
best_nn = NNModel(n_input, best_arch['hidden'], n_classes)
train_model(best_nn, X_train_t, y_train_t, lr=0.01, max_epochs=1000)
preds_best = evaluate_nn(best_nn, X_test_t, y_test_t)

print_metrics(y_te, preds_best, f"Neural Network ({best_model_name})")


---
<a id='regression'></a>
## Section 15 — Linear & Polynomial Regression + Ridge Regularization
**(Labs 9 & 10 — Energy Efficiency / Heating Load)**

### 📖 Theory

**Linear Regression**: fit a straight line y = w₀ + w₁x₁ + ... + wₙxₙ
- OLS (Ordinary Least Squares) minimizes sum of squared residuals

**Polynomial Regression**: add polynomial features x², x³, ... then apply linear regression

**Ridge Regression (L2 regularization)**:
- Adds penalty `α × Σwᵢ²` to the loss to reduce overfitting
- Shrinks coefficients towards 0

### 📐 Formulas
```
OLS cost: J = (1/2n) × Σ(ŷᵢ - yᵢ)²
RMSE: √(MSE) = √((1/n)Σ(ŷᵢ - yᵢ)²)
Ridge cost: J = MSE + α × Σwᵢ²
```

### 🎯 Viva Questions
1. What is overfitting in polynomial regression? How does degree affect it?
2. What does a high training RMSE but low validation RMSE mean?
3. What does ridge regression do to coefficients?
4. What is the "elbow" in the degree vs RMSE plot?

⚠️ **Common mistake:** Using `PolynomialFeatures` on multiple features without Ridge — features explode combinatorially and cause numerical instability (→ need `Ridge(alpha=1e-6)` minimum).


In [ ]:
# ============================================================
# Linear & Polynomial Regression (Labs 9 & 10 pattern)
# Using California Housing as surrogate for Energy Efficiency
# ============================================================

from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_hs = pd.DataFrame(housing.data, columns=housing.feature_names)
y_hs = housing.target  # median house value

# Use only 1 feature first (like Lab 9's x1 approach)
X_hs_1f = X_hs[['MedInc']].values
y_hs_arr = y_hs

X_tr1, X_tmp1, y_tr1, y_tmp1 = train_test_split(X_hs_1f, y_hs_arr, test_size=0.4, random_state=42)
X_vl1, X_te1, y_vl1, y_te1   = train_test_split(X_tmp1,  y_tmp1,   test_size=0.5, random_state=42)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# ---- Simple Linear Regression (1 feature) ----
lr = LinearRegression()
lr.fit(X_tr1, y_tr1)

print("=== Simple Linear Regression (1 feature) ===")
print(f"Intercept  : {lr.intercept_:.4f}")
print(f"Coefficient: {lr.coef_[0]:.4f}")
print(f"Train RMSE : {rmse(y_tr1, lr.predict(X_tr1)):.4f}")
print(f"Val   RMSE : {rmse(y_vl1, lr.predict(X_vl1)):.4f}")
print(f"Test  RMSE : {rmse(y_te1, lr.predict(X_te1)):.4f}")

# Best fit line plot
x1_range = np.linspace(X_tr1.min(), X_tr1.max(), 200).reshape(-1, 1)
plt.figure(figsize=(7, 5))
plt.scatter(X_tr1, y_tr1, alpha=0.3, color='steelblue', s=10, label='Train Data')
plt.plot(x1_range, lr.predict(x1_range), color='red', linewidth=2, label='Best Fit Line')
plt.xlabel('MedInc')
plt.ylabel('House Value')
plt.title('Simple Linear Regression')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---- Polynomial Regression — sweep degrees (Lab 9 pattern) ----
degrees = list(range(2, 9))
train_rmses = []
val_rmses   = []
models_poly = []

for p in degrees:
    poly = PolynomialFeatures(degree=p)
    X_tr_p = poly.fit_transform(X_tr1)
    X_vl_p = poly.transform(X_vl1)
    
    model = LinearRegression()
    model.fit(X_tr_p, y_tr1)
    
    tr_rmse  = rmse(y_tr1, model.predict(X_tr_p))
    vl_rmse  = rmse(y_vl1, model.predict(X_vl_p))
    
    train_rmses.append(tr_rmse)
    val_rmses.append(vl_rmse)
    models_poly.append((model, poly))
    print(f"Degree {p:2d} | Train RMSE: {tr_rmse:.4f} | Val RMSE: {vl_rmse:.4f}")

best_deg = degrees[np.argmin(val_rmses)]
print(f"\nBest degree (by val RMSE): {best_deg}")

# Elbow plot
plt.figure(figsize=(8, 5))
plt.plot(degrees, train_rmses, 'bo-', label='Train RMSE')
plt.plot(degrees, val_rmses,   'rs--', label='Val RMSE')
plt.axvline(best_deg, color='green', linestyle=':', label=f'Best Degree={best_deg}')
plt.xlabel('Polynomial Degree')
plt.ylabel('RMSE')
plt.title('Polynomial Regression: RMSE vs Degree (Elbow Plot)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Ridge Regression (multi-feature, Lab 10 pattern) ----
X_all = X_hs.values
X_tr_all, X_tmp_all, y_tr_all, y_tmp_all = train_test_split(X_all, y_hs_arr, test_size=0.4, random_state=42)
X_vl_all, X_te_all, y_vl_all, y_te_all   = train_test_split(X_tmp_all, y_tmp_all, test_size=0.5, random_state=42)

print("\n=== Ridge vs Linear (Degree 6, All Features) ===")
results_compare = []

for deg in [2, 3, 4]:
    poly = PolynomialFeatures(degree=deg, include_bias=True)
    X_tr_p = poly.fit_transform(X_tr_all)
    X_vl_p = poly.transform(X_vl_all)
    X_te_p = poly.transform(X_te_all)
    
    # No regularization
    lin = Ridge(alpha=1e-9)
    lin.fit(X_tr_p, y_tr_all)
    
    # Ridge
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_tr_p, y_tr_all)
    
    lin_test_rmse   = rmse(y_te_all, lin.predict(X_te_p))
    ridge_test_rmse = rmse(y_te_all, ridge.predict(X_te_p))
    
    print(f"Degree {deg}: Linear RMSE={lin_test_rmse:.4f} | Ridge RMSE={ridge_test_rmse:.4f}")
    results_compare.append({'deg': deg, 'linear': lin_test_rmse, 'ridge': ridge_test_rmse})

print("\n💡 Ridge helps when model overfits — especially at higher polynomial degrees!")


---
<a id='autoregression'></a>
## Section 16 — Autoregression (AR Models) & Neural Networks for Time Series
**(Lab 11 — AAPL Stock / COVID-19)**

### 📖 Theory

**Time Series**: data indexed by time. Key property: past values predict future values.

**Autocorrelation**: correlation of a signal with a delayed copy of itself.
- ACF plot shows if lag-k values are correlated with current value.

**AR(p) Model** (Autoregressive of order p):
```
y(t) = w₀ + w₁y(t-1) + w₂y(t-2) + ... + wₚy(t-p) + ε
```
- p lags used to predict next step
- Coefficients fit via OLS using `statsmodels AutoReg`

**Neural Network for Time Series**: use lag-p values as input features.

### 📐 Metrics for Regression/Forecasting
```
RMSE = √(mean((actual - predicted)²))
MAPE = mean(|actual - predicted| / actual) × 100%
```

### 🎯 Viva Questions
1. What is autocorrelation?
2. What does the ACF plot tell you about optimal lag?
3. How is AR(p) different from linear regression?
4. What is the difference between 1-step ahead and multi-step prediction?
5. Why is MAPE preferred for financial data?


In [ ]:
# ============================================================
# Time Series Analysis — AR Model (Lab 11 pattern)
# Using synthetic sine + noise data (no CSV needed)
# ============================================================

# Generate a stationary-ish time series (like a simplified stock price)
np.random.seed(42)
n = 1000
t = np.arange(n)
series = np.cumsum(np.random.randn(n)) + 50  # Random walk (like stock price)

# --- ACF manually and Pearson correlation for lag-1 ---
df_ts = pd.DataFrame({'y': series})
df_ts['lag1'] = df_ts['y'].shift(1)
df_ts_clean = df_ts.dropna()

from scipy.stats import pearsonr
corr, pval = pearsonr(df_ts_clean['y'], df_ts_clean['lag1'])
print(f"Pearson Correlation (lag-1): {corr:.6f}")
print(f"p-value: {pval:.2e}")

# Line plot
plt.figure(figsize=(12, 4))
plt.plot(series, color='steelblue', linewidth=0.8)
plt.title('Synthetic Time Series (Random Walk)')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.tight_layout()
plt.show()


In [ ]:
# ---- AR(p) Model using statsmodels ----
try:
    from statsmodels.tsa.ar_model import AutoReg
    
    train_size = 800
    train_ts = series[:train_size]
    test_ts  = series[train_size:]
    
    def rmse_ts(actual, predicted):
        return np.sqrt(np.mean((actual - predicted) ** 2))
    
    def mape_ts(actual, predicted):
        return np.mean(np.abs((actual - predicted) / (actual + 1e-8))) * 100
    
    results_ar = []
    for p in [1, 5, 10, 20]:
        model = AutoReg(train_ts, lags=p).fit()
        preds = model.predict(start=train_size, end=train_size + len(test_ts) - 1)
        r = rmse_ts(test_ts, preds[:len(test_ts)])
        m = mape_ts(test_ts, preds[:len(test_ts)])
        results_ar.append({'lag': p, 'RMSE': r, 'MAPE': m})
        print(f"AR(p={p:2d}) | RMSE={r:.4f} | MAPE={m:.4f}%")
    
    print("\n💡 Higher lag generally reduces RMSE for autocorrelated series")
    
except ImportError:
    print("Install statsmodels: pip install statsmodels")


In [ ]:
# ---- NN for Time Series (Lab 11 pattern) ----
def create_lagged_dataset(series, lag):
    """Convert time series to supervised learning dataset."""
    X, y = [], []
    for i in range(lag, len(series)):
        X.append(series[i-lag:i])
        y.append(series[i])
    return np.array(X), np.array(y)


lag = 10
train_size = 800
X_all, y_all = create_lagged_dataset(series, lag)

X_tr_ts = X_all[:train_size - lag].astype(np.float32)
y_tr_ts = y_all[:train_size - lag].astype(np.float32)
X_te_ts = X_all[train_size - lag:].astype(np.float32)
y_te_ts = y_all[train_size - lag:]

# Standardize
sc_X = StandardScaler(); sc_y = StandardScaler()
X_tr_ts_s = sc_X.fit_transform(X_tr_ts)
X_te_ts_s = sc_X.transform(X_te_ts)
y_tr_ts_s = sc_y.fit_transform(y_tr_ts.reshape(-1, 1)).ravel()

# NN for regression (1 output)
class NNRegressor(nn.Module):
    def __init__(self, lag, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(lag, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
nn_reg = NNRegressor(lag, 64)
X_tr_t = torch.tensor(X_tr_ts_s, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr_ts_s, dtype=torch.float32).view(-1, 1)
X_te_t = torch.tensor(X_te_ts_s, dtype=torch.float32)

criterion = nn.MSELoss()
optimizer = optim.SGD(nn_reg.parameters(), lr=0.01, momentum=0.9)
losses_ts = []
for epoch in range(300):
    nn_reg.train()
    optimizer.zero_grad()
    out = nn_reg(X_tr_t)
    loss = criterion(out, y_tr_t)
    loss.backward()
    optimizer.step()
    losses_ts.append(loss.item())

nn_reg.eval()
with torch.no_grad():
    preds_ts = sc_y.inverse_transform(nn_reg(X_te_t).numpy())

rmse_nn = rmse_ts(y_te_ts, preds_ts.ravel())
mape_nn = mape_ts(y_te_ts, preds_ts.ravel())
print(f"\nNN (lag={lag}, hidden=64) | RMSE={rmse_nn:.4f} | MAPE={mape_nn:.4f}%")

# Loss plot
plt.figure(figsize=(8, 4))
plt.plot(losses_ts)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('NN Training Loss (Time Series)')
plt.tight_layout()
plt.show()


---
<a id='kmeans'></a>
## Section 17 — K-Means & K-Medoids Clustering
**(Lab 12 — 6-language audio embeddings, t-SNE features)**

### 📖 Theory

**K-Means**: partition n points into K clusters by minimizing within-cluster variance.

### 📐 Algorithm
```
1. Initialize K centroids randomly
2. Assign each point to nearest centroid (Euclidean distance)
3. Update centroids = mean of all points in cluster
4. Repeat until convergence (centroids don't move)
```

**K-Medoids**: same idea but centroids must be ACTUAL data points (medoids).
- More robust to outliers than K-Means
- Uses `sklearn_extra.cluster.KMedoids`

**Elbow Method**: plot Inertia (within-cluster sum of squares) vs K. Look for the "elbow."

### 📐 Inertia Formula
```
Inertia = Σ_clusters Σ_points ||x - centroid||²
```

### 🎯 Viva Questions
1. How does K-Means choose initial centroids? (random or K-Means++)
2. What is inertia/distortion?
3. What is the elbow method?
4. Why is K-Medoids more robust to outliers?
5. What are the limitations of K-Means? (assumes spherical clusters, requires K)


In [ ]:
# ============================================================
# K-Means Clustering (Lab 12 pattern)
# Using Iris dataset (3 natural clusters, like the 6-language dataset)
# ============================================================

iris_data = load_iris()
X_cluster = iris_data.data[:, :2]  # 2D for visualization
y_true    = iris_data.target

# ---- Purity Score (from scratch — Lab 12 function) ----
def purity_score(y_true, y_pred):
    """
    Purity = Σ_clusters (max_class_count_in_cluster) / total_points
    """
    total_correct = 0
    for cluster_id in np.unique(y_pred):
        mask = (y_pred == cluster_id)
        true_labels_in_cluster = y_true[mask]
        most_common = Counter(true_labels_in_cluster).most_common(1)[0][1]
        total_correct += most_common
    return total_correct / len(y_true)


# ---- K-Means for K = 2 to 10 ----
K_values = list(range(2, 11))
kmeans_results = {'K': [], 'Inertia': [], 'Purity': [], 'NMI': [], 'Silhouette': []}
kmeans_models  = {}

for k in K_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    labels = km.labels_
    kmeans_models[k] = km
    
    inertia  = km.inertia_
    purity   = purity_score(y_true, labels)
    nmi      = normalized_mutual_info_score(y_true, labels)
    sil      = silhouette_score(X_cluster, labels)
    
    kmeans_results['K'].append(k)
    kmeans_results['Inertia'].append(round(inertia, 2))
    kmeans_results['Purity'].append(round(purity, 4))
    kmeans_results['NMI'].append(round(nmi, 4))
    kmeans_results['Silhouette'].append(round(sil, 4))
    
    print(f"K={k}: Inertia={inertia:.2f} | Purity={purity:.4f} | NMI={nmi:.4f} | Sil={sil:.4f}")

kmeans_df = pd.DataFrame(kmeans_results)
print("\n=== K-Means Performance Table ===")
print(kmeans_df.to_string(index=False))


In [ ]:
# ---- Elbow Plot ----
plt.figure(figsize=(8, 5))
plt.plot(kmeans_results['K'], kmeans_results['Inertia'], 'bo-', markersize=8, linewidth=2)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Distortion)')
plt.title('K-Means: Elbow Method — K vs Inertia')
plt.xticks(K_values)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Scatter Plots for K=2,3,4 ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, k in enumerate([2, 3, 4]):
    km = kmeans_models[k]
    labels = km.labels_
    centers = km.cluster_centers_
    axes[idx].scatter(X_cluster[:, 0], X_cluster[:, 1],
                      c=labels, cmap='tab10', alpha=0.6, s=20)
    axes[idx].scatter(centers[:, 0], centers[:, 1],
                      c='black', marker='X', s=200, zorder=5, label='Centroids')
    axes[idx].set_title(f'K-Means (K={k})')
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')
    axes[idx].legend(fontsize=8)
plt.suptitle('K-Means Clustering Results', fontsize=13)
plt.tight_layout()
plt.show()


---
<a id='hierarchical'></a>
## Section 18 — Hierarchical Clustering (Agglomerative + Dendrogram)
**(Lab 13 — 6-language audio embeddings)**

### 📖 Theory

**Agglomerative (Bottom-Up)**:
1. Start: each point is its own cluster
2. Merge the two closest clusters
3. Repeat until one big cluster

**Linkage Criteria** (how to measure cluster distance):
| Linkage | Definition | Characteristics |
|---------|-----------|----------------|
| **Ward** | Minimize within-cluster variance | Compact clusters, most common |
| **Complete** | Max distance between any two points | Compact, equal-sized |
| **Average** | Average distance between all pairs | Balance of single/complete |
| **Single** | Min distance between any two points | Chain effect, elongated clusters |

**Dendrogram**: tree showing the merge hierarchy. Cut at height h → k clusters.

### 🎯 Viva Questions
1. What is the difference between agglomerative and divisive clustering?
2. Which linkage method is best and why? (Usually Ward)
3. How do you determine K from a dendrogram?
4. What is the "chain effect" problem in single linkage?


In [ ]:
# ============================================================
# Hierarchical / Agglomerative Clustering (Lab 13 pattern)
# ============================================================

# Use same 2D Iris data
linkage_methods = ['ward', 'complete', 'average', 'single']
hier_results = {}

print("=== Hierarchical Clustering — Linkage Comparison ===")
print(f"{'Linkage':10s} {'Purity':>10} {'NMI':>10} {'Silhouette':>12}")
print('-' * 46)

for linkage in linkage_methods:
    model = AgglomerativeClustering(n_clusters=3, linkage=linkage)
    labels = model.fit_predict(X_cluster)
    
    purity = purity_score(y_true, labels)
    nmi    = normalized_mutual_info_score(y_true, labels)
    sil    = silhouette_score(X_cluster, labels)
    
    hier_results[linkage] = {'Purity': purity, 'NMI': nmi, 'Silhouette': sil, 'labels': labels}
    print(f"{linkage:10s} {purity:>10.4f} {nmi:>10.4f} {sil:>12.4f}")

# Best linkage
best_linkage = max(linkage_methods,
                   key=lambda lm: hier_results[lm]['Purity'] + 
                                  hier_results[lm]['NMI'] + 
                                  hier_results[lm]['Silhouette'])
print(f"\n✅ Best linkage method: {best_linkage.upper()}")


In [ ]:
# ---- Scatter plots for all 4 linkage methods ----
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, linkage in enumerate(linkage_methods):
    labels = hier_results[linkage]['labels']
    axes[idx].scatter(X_cluster[:, 0], X_cluster[:, 1],
                      c=labels, cmap='tab10', alpha=0.6, s=20)
    axes[idx].set_title(f'Agglomerative — {linkage} linkage\n'
                        f"Purity={hier_results[linkage]['Purity']:.4f} | "
                        f"NMI={hier_results[linkage]['NMI']:.4f}")
    axes[idx].set_xlabel('Feature 1')
    axes[idx].set_ylabel('Feature 2')

plt.suptitle('Hierarchical Clustering: All Linkage Methods', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ---- Dendrogram ----
plt.figure(figsize=(14, 6))

# scipy linkage (for dendrogram) — uses method names directly
Z = sch.linkage(X_cluster, method='ward')
sch.dendrogram(Z, truncate_mode='lastp', p=30,
               leaf_rotation=45, leaf_font_size=10)
plt.title('Dendrogram — Ward Linkage')
plt.xlabel('Sample Index or (Cluster Size)')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

print("💡 Cut the dendrogram horizontally to decide number of clusters.")
print("   The number of vertical lines crossed = number of clusters.")


---
<a id='dbscan'></a>
## Section 19 — DBSCAN (Density-Based Spatial Clustering)
**(Lab 13 — 6-language audio embeddings)**

### 📖 Theory

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise):
- Does NOT require K to be specified
- Finds **arbitrary-shaped clusters**
- Identifies **noise points** (outliers) as label = -1

### 📐 Parameters
- **eps (ε)**: neighborhood radius — if two points within ε, they're neighbors
- **min_samples**: min points within ε to form a core point

### Point Types
- **Core point**: has ≥ min_samples neighbors within ε
- **Border point**: within ε of a core point, but < min_samples neighbors
- **Noise point**: neither core nor border → label = -1

### 🎯 Viva Questions
1. What are the two key parameters of DBSCAN?
2. What is a core point, border point, noise point?
3. Advantage of DBSCAN over K-Means?
4. How do you choose eps? (k-distance graph)
5. Does DBSCAN work well on high-dimensional data?

⚠️ **Common mistake:** Noise points (label=-1) should be handled carefully in metrics — they are included in purity calculation as a separate cluster.


In [ ]:
# ============================================================
# DBSCAN (Lab 13 pattern)
# ============================================================

# Test different eps values
eps_values = [0.3, 0.5, 0.7, 1.0]
min_samples_vals = [5, 5, 5, 5]

print("=== DBSCAN Parameter Sweep ===")
print(f"{'eps':>6} {'min_s':>6} {'#clusters':>10} {'noise%':>8} {'Purity':>8} {'NMI':>8} {'Sil':>8}")
print('-' * 60)

dbscan_results = {}
for eps, min_s in zip(eps_values, min_samples_vals):
    db = DBSCAN(eps=eps, min_samples=min_s)
    labels = db.fit_predict(X_cluster)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = np.sum(labels == -1)
    noise_pct  = n_noise / len(labels) * 100
    
    if n_clusters < 2:
        print(f"{eps:>6.1f} {min_s:>6d} {n_clusters:>10d} {noise_pct:>7.1f}%  (insufficient clusters)")
        continue
    
    purity = purity_score(y_true, labels)  # noise forms its own cluster
    nmi    = normalized_mutual_info_score(y_true, labels)
    
    # Silhouette: only for points that aren't all noise
    non_noise = labels != -1
    if non_noise.sum() > 1 and len(np.unique(labels[non_noise])) > 1:
        sil = silhouette_score(X_cluster[non_noise], labels[non_noise])
    else:
        sil = float('nan')
    
    dbscan_results[eps] = {'labels': labels, 'n_clusters': n_clusters, 
                            'Purity': purity, 'NMI': nmi, 'Sil': sil}
    print(f"{eps:>6.1f} {min_s:>6d} {n_clusters:>10d} {noise_pct:>7.1f}%  "
          f"{purity:>7.4f}  {nmi:>7.4f}  {sil:>7.4f}")


In [ ]:
# ---- Scatter plot for chosen eps ----
eps_show = 0.5  # pick a sensible value
db = DBSCAN(eps=eps_show, min_samples=5)
labels_db = db.fit_predict(X_cluster)

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise    = np.sum(labels_db == -1)

plt.figure(figsize=(8, 6))
unique_labels = sorted(set(labels_db))
cmap = plt.cm.get_cmap('tab10', len(unique_labels))

for i, lbl in enumerate(unique_labels):
    mask = (labels_db == lbl)
    if lbl == -1:
        plt.scatter(X_cluster[mask, 0], X_cluster[mask, 1],
                    c='black', s=10, alpha=0.4, marker='.', label='Noise')
    else:
        plt.scatter(X_cluster[mask, 0], X_cluster[mask, 1],
                    color=cmap(i), s=20, alpha=0.7, label=f'Cluster {lbl}')

plt.title(f'DBSCAN (eps={eps_show}, min_samples=5)\n'
          f'{n_clusters} clusters, {n_noise} noise points')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend(markerscale=2, fontsize=9, ncol=2)
plt.tight_layout()
plt.show()


---
<a id='clustermetrics'></a>
## Section 20 — Clustering Evaluation Metrics
**(Labs 12 & 13)**

### 📐 Metric Reference Table

| Metric | Range | Formula | Description |
|--------|-------|---------|-------------|
| **Purity** | [0, 1] | Σ(majority class per cluster) / N | Higher = better alignment with true labels |
| **NMI** | [0, 1] | MI(Y,C) / √(H(Y)×H(C)) | Normalized mutual info — robust to cluster count |
| **Silhouette** | [-1, 1] | (b-a)/max(a,b) | Cohesion vs separation — higher = better |
| **Inertia** | [0, ∞) | Σ||x-centroid||² | Within-cluster SSS — lower = better |

Where: a = avg distance within cluster, b = avg distance to nearest cluster

### 🎯 Viva Questions
1. What is the silhouette score and what does a negative score mean?
2. Can we use accuracy to evaluate clustering?
3. What is NMI and why is it "normalized"?
4. Can purity always be maximized by making each point its own cluster?

💡 **Tip:** For external metrics (Purity, NMI) you need true labels. For internal metrics (Silhouette, Inertia) you don't.


In [ ]:
# ============================================================
# Clustering Metrics Summary — All 3 Algorithms Compared
# ============================================================

print("=" * 70)
print("FINAL CLUSTERING COMPARISON (K=3, Iris 2D data)")
print("=" * 70)
print(f"{'Algorithm':25s} {'Purity':>8} {'NMI':>8} {'Silhouette':>12}")
print('-' * 60)

# K-Means (K=3)
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km3 = km3.fit_predict(X_cluster)
p_km = purity_score(y_true, labels_km3)
n_km = normalized_mutual_info_score(y_true, labels_km3)
s_km = silhouette_score(X_cluster, labels_km3)
print(f"{'K-Means (K=3)':25s} {p_km:>8.4f} {n_km:>8.4f} {s_km:>12.4f}")

# Agglomerative (best linkage)
labels_hier = hier_results[best_linkage]['labels']
p_h = hier_results[best_linkage]['Purity']
n_h = hier_results[best_linkage]['NMI']
s_h = hier_results[best_linkage]['Silhouette']
print(f"{'Agglomerative (Ward)':25s} {p_h:>8.4f} {n_h:>8.4f} {s_h:>12.4f}")

# DBSCAN
db_final = DBSCAN(eps=0.5, min_samples=5)
labels_db_final = db_final.fit_predict(X_cluster)
n_clust_db = len(set(labels_db_final)) - (1 if -1 in labels_db_final else 0)
if n_clust_db >= 2:
    p_db = purity_score(y_true, labels_db_final)
    n_db = normalized_mutual_info_score(y_true, labels_db_final)
    non_noise = labels_db_final != -1
    s_db = silhouette_score(X_cluster[non_noise], labels_db_final[non_noise]) if non_noise.sum() > 1 else float('nan')
    print(f"{'DBSCAN (eps=0.5)':25s} {p_db:>8.4f} {n_db:>8.4f} {s_db:>12.4f}")
    
print("=" * 60)

# Silhouette over K (for K-Means)
print("\n=== Silhouette Score vs K (K-Means) ===")
for k, sil in zip(kmeans_results['K'], kmeans_results['Silhouette']):
    bar = '█' * int(sil * 20)
    print(f"  K={k:2d}  {sil:.4f}  {bar}")


---
<a id='revision'></a>
## Section 21 — ⏰ 1-Day Revision Strategy (Hour-by-Hour)

### Assumptions:
- Exam is Thursday morning
- You have ~8-10 focused hours on Wednesday
- You've already seen the material; this is a REVISION pass

---

### 📅 Revision Plan

| Time Block | Duration | Topic | Focus |
|-----------|----------|-------|-------|
| **9:00 – 10:00 AM** | 1 hr | Sections 2–4 (EDA, Missing Values, Scaling) | Run code, confirm IQR formula, understand ffill vs impute |
| **10:00 – 11:00 AM** | 1 hr | Sections 5–7 (PCA, Splitting, KNN) | Scree plot, 99% variance trick, K sweep |
| **11:00 – 11:15 AM** | 15 min | ☕ Break | — |
| **11:15 – 12:00 PM** | 45 min | Section 8–9 (Metrics + Helper Functions) | Memorize `print_metrics()` template — it appears in EVERY lab |
| **12:00 – 1:00 PM** | 1 hr | Sections 10–11 (Bayes + Naive Bayes) | Understand BayesClassifier class; Gaussian likelihood |
| **1:00 – 1:45 PM** | 45 min | 🍽️ Lunch | — |
| **1:45 – 2:45 PM** | 1 hr | Sections 12–13 (Logistic Regression + SVM) | 3 variants (original/scaled/PCA), degree sweep |
| **2:45 – 3:30 PM** | 45 min | Section 14 (Perceptron + NN basics) | Understand `model.train()` / `model.eval()` / convergence |
| **3:30 – 3:45 PM** | 15 min | ☕ Break | — |
| **3:45 – 4:45 PM** | 1 hr | Section 16 (Linear/Polynomial Regression + Ridge) | Elbow plot, overfitting, ridge alpha |
| **4:45 – 5:30 PM** | 45 min | Section 17–19 (K-Means, Hierarchical, DBSCAN) | Purity formula, linkage comparison, dendrogram cut |
| **5:30 – 6:30 PM** | 1 hr | Section 20 (Clustering Metrics) | Purity/NMI/Silhouette/Inertia — range + formula |
| **6:30 – 7:00 PM** | 30 min | Quick viva Q&A scan | Read ALL 🎯 viva questions in this notebook |
| **7:00 – 8:00 PM** | 1 hr | Dry run | Pick 2 sections, clear outputs, run from scratch |
| **8:00 PM+** | Rest | Sleep early | Tired brain makes silly mistakes in exams |

---

### 🔑 Most High-Yield Things to Remember

1. **`print_metrics()` template** — used in Labs 6, 7, 8, 13. Know it by heart.
2. **PCA variance threshold** — `PCA(n_components=0.99)` is the pattern.
3. **Fit scaler/PCA on TRAIN only**, transform val+test.
4. **`purity_score()` from scratch** — Counter + most_common — appears in Labs 12 & 13.
5. **Lab 7 structure** = run same code on 3 variants: original, scaled, PCA-All, PCA-99%.
6. **K-Means elbow** = Inertia vs K. **Silhouette** picks the best K.
7. **DBSCAN** = noise points get label -1. `eps` + `min_samples` are the hyperparameters.
8. **Convergence criterion in NN/Perceptron** = `|loss(t) - loss(t-1)| < threshold` for `patience` steps.

### ⚠️ Top 5 Common Exam Mistakes

1. Fitting scaler on full dataset (data leakage)
2. Not calling `model.eval()` during inference in PyTorch
3. Using `accuracy_score` on imbalanced datasets without questioning it
4. Applying PCA before splitting data
5. Forgetting `allow_singular=True` in Bayes classifier's multivariate_normal

### 💡 Top Shortcuts

- `df.isnull().sum()` → missing values per column
- `df.dropna(thresh=n)` → drop rows with fewer than n non-NaN values
- `pd.Series.value_counts()` → class distribution instantly
- `Counter(labels).most_common(1)[0][1]` → get majority class count
- `np.argmax(cumulative >= 0.99) + 1` → number of PCA components for 99% variance
- `KMeans(n_init=10)` → always set `n_init` to avoid warnings
- `zero_division=0` in sklearn metrics → avoids division-by-zero for empty classes

---
**Good luck in your exam, Ayush! 🎯**
